In [2]:
import pandas as pd
import vectorbt as vbt

pairs = ["EURUSD", "AUDNZD", "EURGBP", "AUDCAD", "CHFJPY"]

# Load and combine all CSVs
dfs = {}
for pair in pairs:
    df = pd.read_csv(f"/Users/joaomartins/Desktop/{pair}_daily.csv", index_col=0, parse_dates=True, date_format="%Y-%m-%d")
    df.index.name = "Date"
    dfs[pair] = df

# Build wide-format OHLCV — each column level is a pair
open  = pd.concat({p: dfs[p]["Open"]  for p in pairs}, axis=1)
high   = pd.concat({p: dfs[p]["High"]  for p in pairs}, axis=1)
low    = pd.concat({p: dfs[p]["Low"]   for p in pairs}, axis=1)
close  = pd.concat({p: dfs[p]["Close"] for p in pairs}, axis=1)

# Drop the extra ticker row yfinance sneaks in
for df in [open, high, low, close]:
    df.drop(index="Price", errors="ignore", inplace=True)
    df.drop(index="Ticker", errors="ignore", inplace=True)

# Drop rows where any pair has NaN (weekend gaps, holidays)
close = close.dropna()
open_ = open.reindex(close.index)
high  = high.reindex(close.index)
low   = low.reindex(close.index)

# Now convert safely
high   = high.astype(float)
low    = low.astype(float)
close  = close.astype(float)
open  = open.astype(float)

In [3]:
combined = pd.concat({"Open": open, "High": high, "Low": low, "Close": close}, axis=1)
combined.to_csv("/Users/joaomartins/Desktop/fx_ohlcv_all.csv")

In [4]:
# Save
combined.to_pickle("/Users/joaomartins/Desktop/fx_ohlcv.pkl")

# Reload later — preserves dtypes and index perfectly
combined = pd.read_pickle("/Users/joaomartins/Desktop/fx_ohlcv.pkl")